# 04 · Streaming Data Processing

Live Wikimedia change events go through Kafka and are matched against the IMDb dataset with Spark Structured Streaming.

- **Producer**: `src/streaming_producer.py` (Wikimedia EventStreams → Kafka topic `wiki-changes`, English Wikipedia and Wikidata).
- **Pipeline**: `src/streaming_pipeline.py` (Kafka → stream-static join with an IMDb reference → de-duplication and event-time windows → Parquet).
- **Outputs**: `data/streaming/output/matched_events/`, `data/streaming/output/window_counts/`, `data/streaming/output/metrics/progress.jsonl`.

## What the pipeline does

1. **Reference table (static, Spark batch).** Built from the prepared snapshot and the enrichment outputs of the same snapshot:
   - `wikidata_item`: Wikidata items (`Q…`) of the enriched films and principals;
   - `wikipedia_page`: English Wikipedia articles of the enriched films and principals;
   - `imdb_title`: base titles of IMDb movies released 1980–2025, matched against articles named `<title> (<year> film)`.
2. **Matching (stream-static join).** Each event yields candidate keys from its wiki and title; candidates are joined with the broadcast reference.
3. **Event time.** `event_time` is the Wikimedia `timestamp`; `ingested_at` is set by the producer and `kafka_timestamp` by Kafka, which gives the ingestion delay.
4. **Late data.** A watermark of 10 minutes (`STREAM_WATERMARK`) bounds the two stateful operators: `dropDuplicatesWithinWatermark` on matched events and 5-minute event-time window counts (`append` mode, a window is written once the watermark passes its end). Rows older than the watermark are dropped by these operators and counted in `numRowsDroppedByWatermark`.
5. **Metrics.** A `StreamingQueryListener` appends every micro-batch progress report to `metrics/progress.jsonl`.

In [ ]:
import shutil
import subprocess

if shutil.which("docker"):
    result = subprocess.run(["docker", "ps", "--filter", "name=kafka"], capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    print("docker CLI not found: check that the Kafka broker is reachable at KAFKA_BROKER.")

## Running the pipeline

Kafka must be running (`docker compose up -d`) and a prepared snapshot must exist (notebook 01; notebook 03 for the Wikidata/Wikipedia matches). Run from the repository root, in two terminals:

```bash
# terminal 1 — producer (real contact required by Wikimedia)
export WIKI_CONTACT="<group e-mail>"
python -m src.streaming_producer

# terminal 2 — pipeline, stopped automatically after 30 minutes
export STREAM_DURATION_MINUTES=30
python -m src.streaming_pipeline
```

To check the late-data handling, publish a synthetic event 30 minutes in the past once the pipeline has processed a few batches:

```bash
python -m src.streaming_producer --late-probe "Inside Out (2015 film)" --minutes 30
```

In [ ]:
import json
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next((path for path in (cwd, *cwd.parents) if (path / "src/imdb_loader.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run the notebook from the IMDb repository or its notebooks folder.")
sys.path.insert(0, str(ROOT))

from pyspark.sql import functions as F
from src.imdb_loader import data_root, spark_session

OUTPUT = data_root(ROOT) / "streaming" / "output"
spark = spark_session("Streaming results")
matched = spark.read.parquet((OUTPUT / "matched_events").as_posix())
windows = spark.read.parquet((OUTPUT / "window_counts").as_posix())
progress = [json.loads(line) for line in (OUTPUT / "metrics" / "progress.jsonl").read_text(encoding="utf-8").splitlines()]
batches = [record["progress"] for record in progress if record["event"] == "progress"]
print(f"Matched event rows: {matched.count():,} · window rows: {windows.count():,} · progress reports: {len(batches):,}")

## Streaming metrics

In [ ]:
from datetime import datetime

summary = []
for name in sorted({batch["name"] for batch in batches}):
    runs = [batch for batch in batches if batch["name"] == name]
    started, ended = runs[0]["timestamp"], runs[-1]["timestamp"]
    duration_s = (datetime.fromisoformat(ended) - datetime.fromisoformat(started)).total_seconds()
    rows_in = sum(batch["numInputRows"] for batch in runs)
    dropped = sum(op.get("numRowsDroppedByWatermark", 0) for batch in runs for op in batch["stateOperators"])
    summary.append((name, started, ended, round(duration_s / 60, 1), len(runs), rows_in,
                    round(rows_in / duration_s, 2) if duration_s else None, dropped, runs[-1]["eventTime"].get("watermark")))
spark.createDataFrame(summary, "query string, first_batch string, last_batch string, minutes double, batches long, "
                      "kafka_rows_read long, rows_per_second double, dropped_by_watermark long, last_watermark string").show(truncate=False)

In [ ]:
print("Matched events by type")
matched.groupBy("match_type").agg(
    F.countDistinct("event_uid").alias("events"), F.countDistinct("match_key").alias("distinct_items"),
    F.count_if(F.coalesce("_synthetic", F.lit(False))).alias("synthetic"),
).orderBy(F.desc("events")).show(truncate=False)

print("Ingestion delay (seconds, real events)")
matched.filter(~F.coalesce("_synthetic", F.lit(False))).select("event_uid", "ingest_delay_s").distinct().agg(
    F.count("*").alias("events"),
    F.percentile_approx("ingest_delay_s", 0.5).alias("median"),
    F.percentile_approx("ingest_delay_s", 0.95).alias("p95"),
    F.max("ingest_delay_s").alias("max"),
).show()

print("Most edited IMDb items")
matched.groupBy("match_type", "label", "imdb_ids").agg(F.countDistinct("event_uid").alias("events")).orderBy(
    F.desc("events")).show(15, truncate=60)

matched.select("event_time", "wiki", "type", "title", "match_type", "imdb_ids").orderBy(F.desc("event_time")).show(15, truncate=50)

In [ ]:
print("Matched events per 5-minute event-time window (written once the watermark has passed)")
windows.orderBy("window_start", "match_type").show(50, truncate=False)

late = matched.filter(F.col("event_uid").startswith("late-probe-"))
print(f"Late probes that reached the output: {late.select('event_uid').distinct().count()} "
      "(expected 0 when the probe is sent after the watermark has advanced)")

In [ ]:
spark.stop()